<a href="https://colab.research.google.com/github/vivaan3141/UC-Dashboard-Construction-Vivaan-Gupta/blob/main/UC-Dashboard-GoogleColab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#The question:
## In Fall 2025, how significantly do 25th percentile admit GPA thresholds and admit rate penalties vary for Computer Science across all 9 UC undergraduate campuses compared to overall campus averages?

In [3]:
import pandas as pd
import numpy as np

# Load dataset
try:
    df_disc = pd.read_csv('uc_freshman_admission_by_discipline.csv')
except FileNotFoundError:
    df_disc = pd.read_csv('Data/uc_freshman_admission_by_discipline.csv')

# Dynamically locate column names
campus_col = [c for c in df_disc.columns if 'camp' in c.lower()][0]
disc_col = [c for c in df_disc.columns if any(k in c.lower() for k in ['disc', 'major', 'broad'])][0]
app_col = [c for c in df_disc.columns if 'app' in c.lower() and 'gpa' not in c.lower()][0]
adm_col = [c for c in df_disc.columns if 'adm' in c.lower() and 'gpa' not in c.lower() and 'rate' not in c.lower()][0]

p25_col = [c for c in df_disc.columns if '25' in c and any(k in c.lower() for k in ['adm', 'gpa'])][0]
p75_col = [c for c in df_disc.columns if '75' in c and any(k in c.lower() for k in ['adm', 'gpa'])][0]

# Convert numeric columns safely
for col in [app_col, adm_col, p25_col, p75_col]:
    if df_disc[col].dtype == object:
        df_disc[col] = df_disc[col].astype(str).str.replace(',', '').str.strip().astype(float)

# Exclude systemwide aggregates
campuses_df = df_disc[~df_disc[campus_col].str.contains('systemwide|universitywide', case=False, na=False)].copy()
campuses_df['admit_rate'] = campuses_df[adm_col] / campuses_df[app_col]

# Compute campus-wide average admit rates
campus_totals = campuses_df.groupby(campus_col)[[adm_col, app_col]].sum()
campus_overall = (campus_totals[adm_col] / campus_totals[app_col]).rename('Overall Admit Rate')

# Filter for Computer Science
cs_df = campuses_df[campuses_df[disc_col].str.contains('Computer Science', case=False, na=False)].set_index(campus_col)

# Summary table
summary = pd.DataFrame({
    'Overall Admit Rate': campus_overall,
    'CS Admit Rate': cs_df['admit_rate'],
    'CS Admit Penalty': campus_overall - cs_df['admit_rate'],
    'CS 25th GPA': cs_df[p25_col],
    'CS 75th GPA': cs_df[p75_col],
    'CS IQR': cs_df[p75_col] - cs_df[p25_col]
}).sort_values('CS Admit Penalty', ascending=False)

print("--- FALL 2025 COMPUTER SCIENCE SELECTIVITY SUMMARY ---")
print(summary)

--- FALL 2025 COMPUTER SCIENCE SELECTIVITY SUMMARY ---
               Overall Admit Rate  CS Admit Rate  CS Admit Penalty  \
campus                                                               
Davis                    0.442988       0.193337          0.249651   
San Diego                0.281258       0.198702          0.082556   
Riverside                0.865243       0.811168          0.054075   
Berkeley                 0.113222       0.064513          0.048710   
Santa Barbara            0.382072       0.339130          0.042941   
Los Angeles              0.095911       0.073203          0.022708   
Irvine                   0.293523       0.275832          0.017692   
Santa Cruz               0.725122       0.793851         -0.068728   
Merced                   0.943261            NaN               NaN   

               CS 25th GPA  CS 75th GPA  CS IQR  
campus                                           
Davis                 4.20         4.30    0.10  
San Diego             4.